# Benchmarking with MovieLens dataset

1. Popularity model as basline
2. Collaborative filtering
3. 

In [78]:
import torch as pt
from torch.utils.data import Dataset, DataLoader
from os.path import join
from collections import defaultdict
import random
import math

pt.manual_seed(42)

DATASET = "ml-32m"
DATA_PATH_RAW = join('.', 'data', "raw", DATASET)
DATA_PATH_PRE = join('.', 'data', "preprocessed", DATASET)

In [79]:
# load data tensors
X_train = pt.load(join(DATA_PATH_PRE, 'X_train.pt'))
X_test = pt.load(join(DATA_PATH_PRE, 'X_test.pt'))

print("Shape (n_users, n_items):", "\n       ", X_train.shape, "\n")
print("Shape (n_users, n_items):", "\n       ", X_test.shape, "\n")

Shape (n_users, n_items): 
        torch.Size([198979, 55173]) 

Shape (n_users, n_items): 
        torch.Size([198979, 55173]) 



In [80]:
# Access indices and values of the sparse COO tensor X_train
indices = X_train.indices()
values = X_train.values()

print("Indices shape:", indices.shape)
print("Values shape:", values.shape[0])
print("First few indices:", indices[:, :5])
print("First few values:", values[:5])

Indices shape: torch.Size([2, 11874951])
Values shape: 11874951
First few indices: tensor([[0, 0, 0, 0, 0],
        [0, 1, 2, 3, 4]])
First few values: tensor([1., 1., 1., 1., 1.])


In [81]:
# wrap data in Dataset class
class RatingsDataset(Dataset):
    """Dataset wrapping sparse COO tensor of user-item interactions.

    The dataset is indexed by user and returns the dense user-item interaction
    vector for each user.
    
    """
    def __init__(self, X):
        self.X = X
        self.num_users, self.num_items = X.shape

    def __len__(self):
        return self.num_users

    def __getitem__(self, user_idx):
        return self.X[user_idx].to_dense()

In [82]:
# create Dataset instances
train_dataset = RatingsDataset(X_train)
test_dataset = RatingsDataset(X_test)

print("First user-item interaction vector:\n", train_dataset[0])

First user-item interaction vector:
 tensor([1., 1., 1.,  ..., 0., 0., 0.])


In [83]:
# create DataLoader instances
BATCH_SIZE = 256
train_dataloader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_dataloader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

## 1. Popularity Model
This model simply ranks all of the items by their popularity, i.e. the number of user interactions. This is the baseline model, all other models should be better than this.

In [84]:
popularity = pt.sparse.sum(X_train, dim=0).to_dense()
print("Item popularity vector shape:        ", popularity.shape)
print("First 10 item popularity values:     ", popularity[:5])
print("Number of unpopular items (0 interactions):", pt.sum(popularity == 0).item())

Item popularity vector shape:         torch.Size([55173])
First 10 item popularity values:      tensor([20078.,  5350., 28040.,  3913.,   573.])
Number of unpopular items (0 interactions): 11275


The popularity array holds the number of positive interactions, which we can rank to find the most popular movies in the training set. We use this ranking to predict that these movies will also be the most popular in the test. Popularity models are based on the simple rationale:

“If I know absolutely nothing about this user, what should I show?”

This is still a powerful baseline because real datasets have blockbusters and viral content and it makes sens to base a good amount of recommendations on global popularity.

Therefore, they are used for:
- cold start (either of new users or the whole application)
- fallback mode
- sanity check (every more complex ML model must beat it)

In [85]:
# now we want to evaluate the popularity model with the test set using different metrics, for now let's compute Recall@K / NDCG@K

# Convert sparse tensors to COO format (if not already in that format) so that there are no duplicate indices
train_coo = X_train.coalesce()
test_coo  = X_test.coalesce()

# Extract user and item indices from the COO tensors for train and test sets
# all are (num_interactions)
train_users, train_items = train_coo.indices()[0], train_coo.indices()[1]
test_users, test_items = test_coo.indices()[0], test_coo.indices()[1]

In [86]:
# Build lookup: which items each user interacted with
# for train set is used as it is only relevant whether the user interacted with the item or not
# set is O(1) lookup while list is O(n)
train_items_by_user = defaultdict(set)
for u, i in zip(train_users.tolist(), train_items.tolist()):
    train_items_by_user[u].add(i)

test_items_by_user = defaultdict(list)
for u, i in zip(test_users.tolist(), test_items.tolist()):
    test_items_by_user[u].append(i)

train_items_by_user[999]  # Example: items interacted by user 999

{176, 710, 715, 1024, 1496, 2329, 3037}

In [ ]:
# TODO make faster by vectorizing
def evaluate_popularity(popularity, train_items_by_user, test_items_by_user,
                        num_items, K=10, num_negatives=100):

    recalls = []
    ndcgs = []

    for u in test_items_by_user:

        # pick ONE true test item (you can also loop over all)
        true_item = random.choice(test_items_by_user[u])

        # sample negatives
        negatives = []
        while len(negatives) < num_negatives:
            j = random.randint(0, num_items - 1)
            if j not in train_items_by_user[u] and j != true_item:
                negatives.append(j)

        candidates = [true_item] + negatives

        # score by popularity
        scores = popularity[candidates]

        # rank (descending)
        ranked = pt.argsort(scores, descending=True)

        # find rank of true item
        rank = (ranked == 0).nonzero(as_tuple=True)[0].item() + 1  # +1 for 1-based

        # Recall@K
        recalls.append(1 if rank <= K else 0)

        # NDCG@K
        if rank <= K:
            ndcgs.append(1.0 / math.log2(rank + 1))
        else:
            ndcgs.append(0.0)

    return sum(recalls)/len(recalls), sum(ndcgs)/len(ndcgs)

In [92]:
popularity[[0,5,999]]

tensor([20078., 13586.,   277.])

In [ ]:
# let's evaluate the popularity model
recall_at_K, ndcg_at_K = evaluate_popularity(
    popularity,
    train_items_by_user,
    test_items_by_user,
    num_items=X_train.shape[1],
    K=10,
    num_negatives=100
)

# TODO: way too good results, check correctness 
print(f"Popularity Model - Recall@10: {recall_at_K:.4f}, NDCG@10: {ndcg_at_K:.4f}")

Popularity Model - Recall@10: 0.9841, NDCG@10: 0.8007
